# Sky source review

Interactive review of a sky position in OVRO-LWA time–frequency data. Enter a
**coordinate string** in the review UI — ICRS degrees (`RA, Dec`) or a source name —
then use the action buttons:

- **Center** — centers HiPS and the radio overlay on the coordinate field (typed
  name or sky click). Use after panning when you want to return to a named target.
- **Generate heatmap** — build the time–frequency heatmap for the coordinate.
- **Overlay: on/off** — toggle the radio overlay. Off hides it (HiPS only); clicking
  a heatmap cell turns the overlay back on and loads that slice. The heatmap stays
  clickable either way.

A **zeros heatmap grid** (full Zarr `time × frequency` shape) appears as soon as the
store opens, so you can **click any cell to load that slice as an overlay** centered
on the current coordinate — no need to **Generate heatmap** first. Generating a
heatmap simply replaces the zeros with computed values.

While typing a name (first character is a letter), a dropdown of matching entries from
`known_sources.yaml` appears; pick one or keep typing (tab completion logs RA/Dec).
Numeric-first input is RA/Dec.

For the active coordinate and **heatmap method**:

1. Build a time × frequency map (tracked pixel, patch statistic, patch maximum, or
   Gaussian patch fit — same options as `jupiter_flux_review.ipynb`, plus `mad`, `std`,
   `mean`, `min`).
2. **Click** a cell in the heatmap to overlay that Zarr slice in **astrowidget.SkyWidget**
   at the current view center (view-locked reprojection when HiPS is active; zoom and
   pan are preserved when changing time/frequency). Pan to align HiPS with the radio
   layer; click **Center** to put the catalog source in the middle of the field
   (keeps the overlay; resets the heatmap only when moving away from a **computed**
   target with no overlay loaded).

**Stokes I/V** — when the Zarr has both polarizations, use the Stokes toggle for
overlay, heatmap, and patch fit. After switching Stokes, press **Generate heatmap**
so the time–frequency map matches; heatmap cell clicks load the active Stokes.

Launch with: `pixi run jupyter lab`

**Run cells in order** (config → imports → optional Dask → launch UI). Restart the
kernel after upgrading `ovro_lwa_portal` or `astrowidget` (editable install).


In [ ]:
# Edit before running if your paths or cuts differ.
import os
from pathlib import Path

## Checked again when the review UI starts; fix typos before launching.
#ZARR_PATH = Path("/fast/claw/V-10min-Taper-Robust-0-dec24.zarr/")
ZARR_PATH = Path("/fast/claw/IV-10min-Taper-Robust-0_2024only.zarr/")

# Optional default for the Coordinate field (edit in the review UI after launch).
COORDINATE_STRING = ""

# Fall back to NED ObjectLookup when SkyCoord.from_name fails (requires network).
USE_NED_FALLBACK = True
NED_TIMEOUT_S = 10.0

# Known source names for autocomplete. Resolved relative to cwd and notebooks/.
KNOWN_SOURCES_PATH = Path("known_sources.yaml")

PATCH_SCALE = 5.0  # patch half-width = ceil(scale * max beam FWHM in pixels)
SKY_FOV_DEG = 8.0

# Default heatmap fill: tracked pixel or patch statistics (patch_max, mad, std, …)
HEATMAP_METHOD = "dynamic_spectrum"
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 10.0

# Zarr read pattern (matches jupiter_flux_review.ipynb)
ZARR_LM_CHUNK = 512

# Set False to scan the image centre for the first finite time index (extra I/O)
SKIP_FIRST_VALID_SKY_SCAN = True

# Optional local Dask cluster — see Notes (Parallelism / Dask).
# Enable for patch heatmaps on large incremental stores (I-deep, 100+ times).
# Leave False for QA subsets and for dynamic_spectrum-only review.
USE_DASK_CLIENT = False
DASK_WORKERS = 6
DASK_THREADS_PER_WORKER = 1  # use 1 when DASK_PROCESSES=True (CPU workers)
DASK_MEMORY_LIMIT = "16GiB"
DASK_PROCESSES = True  # True for CPU-bound patch reduce; False for I/O-debug only

# HiPS calibration sky (Aladin Lite background). Served by the ovro-lwa-portal Jupyter
# server extension at HIPS_HTTP_PREFIX (default /calibration/hips). Restart Jupyter after
# upgrading the package if HiPS tiles 404.
HIPS_ROOT = Path("/lustre/pipeline/calibration/hips")
HIPS_BACKGROUND = HIPS_ROOT / "Blue_I_deep_Taper_Robust-0.75_Jan25.hips"
# Override with full URL if tiles are served elsewhere (set OVRO_HIPS_HTTP_BASE too).
HIPS_HTTP_PREFIX = os.environ.get("OVRO_HIPS_HTTP_BASE", "/calibration/hips")
os.environ.setdefault("OVRO_HIPS_HTTP_BASE", HIPS_HTTP_PREFIX)
os.environ.setdefault("OVRO_HIPS_ROOT", str(HIPS_ROOT))

# HiPS background display scaling (Aladin Lite setCuts).
HIPS_BACKGROUND_PERCENTILE_LOW = 1.0
HIPS_BACKGROUND_PERCENTILE_HIGH = 99.0
# Optional fixed HiPS cuts (set both to override percentile sampling).
BACKGROUND_CUT_MIN = None
BACKGROUND_CUT_MAX = None
BACKGROUND_OPACITY = 1.0

# Radio overlay color scale (SkyWidget WebGL layer).
OVERLAY_COLORMAP = "magma"  # inferno, viridis, plasma, magma, grayscale
OVERLAY_STRETCH = "log"  # linear, log, sqrt, asinh
OVERLAY_PERCENTILE_LOW = 1.0
OVERLAY_PERCENTILE_HIGH = 99.9
# Optional fixed overlay vmin/vmax (set both to override per-slice percentiles).
OVERLAY_VMIN = None
OVERLAY_VMAX = None
OVERLAY_OPACITY = 1.0



In [ ]:
import ovro_lwa_portal as ovro
from ovro_lwa_portal.viz.source_review_app import (
    SourceReview,
    SourceReviewConfig,
    configure_source_review_notebook,
)

configure_source_review_notebook()


In [ ]:
# Run before the launch cell so get_client() is registered during heatmap generate.
if USE_DASK_CLIENT:
    from dask.distributed import Client, get_client

    try:
        dask_client = get_client()
    except ValueError:
        dask_client = Client(
            n_workers=DASK_WORKERS,
            threads_per_worker=DASK_THREADS_PER_WORKER,
            processes=DASK_PROCESSES,
            memory_limit=DASK_MEMORY_LIMIT,
        )
    print(dask_client)
    print(f"Dashboard: {dask_client.dashboard_link}")
else:
    print(
        "Dask Client disabled — radport uses threaded Zarr I/O in the kernel and "
        "a local process pool for patch_max / mad / std reductions when no Client "
        "is active."
    )


In [ ]:
ovro.validate_local_zarr_store(ZARR_PATH)

review = SourceReview(
    ZARR_PATH,
    coordinate_string=COORDINATE_STRING,
    known_sources_path=KNOWN_SOURCES_PATH,
    patch_scale=PATCH_SCALE,
    sky_fov_deg=SKY_FOV_DEG,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
    heatmap_method=HEATMAP_METHOD,
    config=SourceReviewConfig(
        zarr_lm_chunk=ZARR_LM_CHUNK,
        polarization_isel=(0, 1),
        skip_first_valid_sky_scan=SKIP_FIRST_VALID_SKY_SCAN,
        use_ned_fallback=USE_NED_FALLBACK,
        ned_timeout_s=NED_TIMEOUT_S,
        hips_root=HIPS_ROOT,
        hips_background=HIPS_BACKGROUND,
        hips_http_prefix=HIPS_HTTP_PREFIX,
        hips_background_percentile_low=HIPS_BACKGROUND_PERCENTILE_LOW,
        hips_background_percentile_high=HIPS_BACKGROUND_PERCENTILE_HIGH,
        background_cut_min=BACKGROUND_CUT_MIN,
        background_cut_max=BACKGROUND_CUT_MAX,
        background_opacity=BACKGROUND_OPACITY,
        overlay_colormap=OVERLAY_COLORMAP,
        overlay_stretch=OVERLAY_STRETCH,
        overlay_percentile_low=OVERLAY_PERCENTILE_LOW,
        overlay_percentile_high=OVERLAY_PERCENTILE_HIGH,
        overlay_vmin=OVERLAY_VMIN,
        overlay_vmax=OVERLAY_VMAX,
        overlay_opacity=OVERLAY_OPACITY,
    ),
    validate_zarr=False,
)
review.panel


## Notes

- **Coordinate** — enter `"RA_deg, Dec_deg"` or a source name. When the first
  character is a letter, an `includes` dropdown lists hits from `known_sources.yaml` (pick one or
  keep typing; tab completion logs RA/Dec). Click **Center** to slew HiPS and reproject the
  overlay onto the field coordinate (overlay is **not** cleared). **Pan** adjusts HiPS alignment —
  use Center, not pan, to put a catalog source in the middle. Names resolve via
  `SkyCoord.from_name`, then NED ObjectLookup when enabled. Parse failures log a **WARNING**
  in the activity log.
- **Stokes** — I/V radio when the store has multiple polarizations (`polarization_isel`
  in config limits which Stokes appear). Switching Stokes updates overlay/heatmap
  intent; press **Generate heatmap** to refresh the spectrum. Activity log and overlay
  status include `Stokes`, `t=`, and `f=`.
- **Overlay toggle** — **Overlay: on/off** hides the radio layer (HiPS only). Clicking
  a heatmap cell **turns the overlay on** and loads that slice (view-locked pan/zoom).
- **Click any cell** — even on the initial zeros grid, a click loads that Zarr slice
  as an overlay centered on the current coordinate. If no coordinate is set yet, the
  status prompts you to enter one (it resolves the field automatically on the click).
- **Gray heatmap cells** — NaN when the target is **outside this snapshot's field of view** at
  that time, when SKY is masked at the tracked pixel/patch, when the `(time, frequency)` cell is an
  **empty subband slot** (no data for that frequency in ingest), or when **Fit overlay** rejects
  the fit by the χ² cut. The activity log prints a footprint hint when many cells are missing.
- **Package upgrades** — confirm `ovro_lwa_portal.__file__` points at `src/ovro_lwa_portal/`
  after viz or accessor changes; restart the kernel. Clear notebook outputs before commit
  (large embedded comm state causes stale **model not found** after reopen).
- **Patch size** — `PATCH_SCALE` × max beam FWHM at each time step.
- **Zarr open** — `open_dataset(..., chunks="auto").chunk({"l": 512, "m": 512})` like `jupiter_flux_review.ipynb` (`ZARR_LM_CHUNK`). Set `SKIP_FIRST_VALID_SKY_SCAN=False` only if you need a centre-pixel time scan on open.
- **Progress** — dynamic spectrum reports **Pixel track** then **Pixel I/O** (per-time reads); patch methods report in the activity log (`Patch I/O`, then `Statistics` or `Patch fit`) with time-step counts.
- First extraction on a full cube can take tens of seconds per coordinate depending on Zarr I/O and time-axis length.
- **Sky background** — OVRO-LWA calibration HiPS (`HIPS_BACKGROUND`) loads by default via
  `SkyWidget.background_survey`. Set `OVRO_HIPS_HTTP_BASE` (or edit `HIPS_HTTP_BASE`) so the
  browser can fetch tiles; Aladin Lite cannot read `/lustre/...` directly. With HiPS active,
  `overlay_view_lock` reprojects the radio overlay to the current view after pan/zoom; heatmap
  load and tap use the panned view center (catalog coordinates still drive heatmap extraction).
  Click a heatmap cell to overlay the Zarr radio slice on the background.
- **Parallelism / Dask (optional)** — OVRO-LWA stores use per-time WCS, so patch heatmaps read a different pixel window each time step. With an active `Client`, patch **extract** (fused Zarr I/O) and per-time **reduce** both run on **Dask workers** (`_patch_extract_scheduler` / `_patch_reduce_scheduler`). Without a Client, extract uses threaded reads in the kernel and reduce uses a local **process** pool. `dynamic_spectrum` always stays in the kernel (no Client needed).

  | Heatmap method | Large store (I-deep) | QA / small subset |
  | --- | --- | --- |
  | `dynamic_spectrum` | Leave `USE_DASK_CLIENT = False` | Leave **off** |
  | `patch_max`, `mad`, `std`, `mean`, `min` | **`USE_DASK_CLIENT = True`**, `DASK_PROCESSES = True` | Optional off (local processes pool is fine) |

  **Order:** run the Dask cell **before** `review = SourceReview(...)` so `get_client()` is registered when you **Generate heatmap**. The activity log also prints the dashboard URL on panel load when a Client is active.

  **Chunking:** keep `open_dataset(..., chunks="auto").chunk({"l": ZARR_LM_CHUNK, "m": ZARR_LM_CHUNK})` — do not add `time: 1` rechunk under an active Client on large incremental stores.

  **Memory:** if workers spill, lower `DASK_WORKERS` or raise `DASK_MEMORY_LIMIT`. Fused patch I/O reduces graph overhead but peak memory is still ~one batch of patches.

  **Override:** set `OVRO_RADPORT_PATCH_SCHEDULER` in the environment for advanced local (no-Client) scheduler tuning.
